# Training Notebook (Colab / Local)

Config-driven training pipeline. Core logic lives in `data_splitter.py`, `dataset.py`,
`model.py`, `pytorch_lightning.py`, and `training_utils.py` — this notebook just wires
them together against `configs/base.yaml`. Edit those files directly; local runs (no
`google.colab` import) pick up changes immediately, no push/pull needed.

**Kaggle auth** (only needed if `data/` isn't already present): tries, in order, an
existing `KAGGLE_API_TOKEN` env var, `~/.kaggle/access_token`, `~/.kaggle/kaggle.json`,
a Colab secret named `KAGGLE_API_TOKEN` (browser UI only), then an interactive prompt.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Works around a known Windows conda/pip OpenMP DLL conflict (harmless elsewhere).
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

GIT_URL = 'https://github.com/hagairavid18/beilinson.git'
GIT_BRANCH = 'main'

try:
    import google.colab  # noqa: F401
    PROJECT_ROOT = Path('/content/beilinson')
    if PROJECT_ROOT.exists():
        subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'pull'])
    else:
        subprocess.check_call(['git', 'clone', '--branch', GIT_BRANCH, GIT_URL, str(PROJECT_ROOT)])
except ImportError:
    # Not on Colab (e.g. a local kernel) - use the repo checkout we're already in.
    PROJECT_ROOT = Path.cwd()

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
(PROJECT_ROOT / 'data').mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / 'artifacts').mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Has data already:', any((PROJECT_ROOT / 'data').glob('*/*')))

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

In [5]:
import os

data_dir = PROJECT_ROOT / 'data'

if any(data_dir.glob('*/*')):
    print('Dataset already present, skipping download.')
else:
    import getpass
    import kagglehub

    access_token_path = Path.home() / '.kaggle' / 'access_token'
    kaggle_json_path = Path.home() / '.kaggle' / 'kaggle.json'

    if os.environ.get('KAGGLE_API_TOKEN') or access_token_path.exists() or kaggle_json_path.exists():
        print('Using existing local Kaggle credentials.')
    else:
        try:
            from google.colab import userdata
            os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
            print('Using Kaggle token from Colab secrets.')
        except Exception:
            # userdata secrets need the actual Colab browser UI; when the kernel is
            # driven from VS Code (or there's no KAGGLE_API_TOKEN secret), fall back
            # to an interactive prompt. Nothing typed here is written to disk.
            os.environ['KAGGLE_API_TOKEN'] = getpass.getpass('Kaggle API token: ')

    print('Downloading dataset from Kaggle...')
    cache_path = Path(kagglehub.dataset_download('hasyimabdillah/workoutexercises-images'))

    # kagglehub caches the dataset under its own path; if it's wrapped in one extra
    # top-level folder, look inside that instead of the cache root.
    source = cache_path
    entries = list(source.iterdir())
    if len(entries) == 1 and entries[0].is_dir():
        source = entries[0]

    for class_dir in source.iterdir():
        if class_dir.is_dir():
            target = data_dir / class_dir.name
            if not target.exists():
                target.symlink_to(class_dir, target_is_directory=True)

class_dirs = sorted(p.name for p in data_dir.iterdir() if p.is_dir())
print(f'{len(class_dirs)} classes found:', class_dirs)

Dataset already present, skipping download.
22 classes found: ['barbell biceps curl', 'bench press', 'chest fly machine', 'deadlift', 'decline bench press', 'hammer curl', 'hip thrust', 'incline bench press', 'lat pulldown', 'lateral raises', 'leg extension', 'leg raises', 'plank', 'pull up', 'push up', 'romanian deadlift', 'russian twist', 'shoulder press', 'squat', 't bar row', 'tricep dips', 'tricep pushdown']


## Run

In [ ]:
import yaml

from training_utils import run_training

with open(PROJECT_ROOT / 'configs' / 'base.yaml', 'r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)

CONFIG

In [ ]:
results = run_training(CONFIG, PROJECT_ROOT)
results

In [ ]:
import json

summary_path = PROJECT_ROOT / 'artifacts' / 'training_summary.json'
with open(summary_path, 'r', encoding='utf-8') as handle:
    summary = json.load(handle)

summary